In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [26]:
!pip install transformers datasets peft scikit-learn -q

In [27]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset("glue", "cola")
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['sentence'], truncation=True, padding='max_length', max_length=128)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print("Data ready!")
print(tokenized["train"][0].keys())

Data ready!
dict_keys(['labels', 'input_ids', 'attention_mask'])


In [28]:
import time
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import matthews_corrcoef
from datasets import load_dataset

# Use BERT instead
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

dataset = load_dataset("glue", "cola")

def tokenize(batch):
    return tokenizer(batch['sentence'], truncation=True, padding='max_length', max_length=128)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "key", "value"]
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    mcc = matthews_corrcoef(labels, predictions)
    return {"matthews_correlation": mcc}

training_args = TrainingArguments(
    output_dir="./lora_cola",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.01,
)

start_time = time.time()
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
lora_time = time.time() - start_time
print(f"\nLoRA training time: {lora_time:.1f}s")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 443,906 || all params: 109,927,684 || trainable%: 0.4038


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Matthews Correlation
1,0.917290,0.922094,0.456965
2,0.838128,0.940623,0.472159
3,0.813156,0.943101,0.477570


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



LoRA training time: 305.3s


In [29]:
labels = [ex['labels'].item() for ex in tokenized['train']]
print(f"Class 0 (ungrammatical): {labels.count(0)}")
print(f"Class 1 (grammatical): {labels.count(1)}")

Class 0 (ungrammatical): 2528
Class 1 (grammatical): 6023


In [30]:
from peft import AdaLoraConfig

base_model2 = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

adalora_config = AdaLoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query", "key", "value"],
    total_step=1000,
)

adalora_model = get_peft_model(base_model2, adalora_config)
adalora_model.print_trainable_parameters()

training_args2 = TrainingArguments(
    output_dir="./adalora_cola",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.01,
)

start_time = time.time()
trainer2 = Trainer(
    model=adalora_model,
    args=training_args2,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer2.train()
adalora_time = time.time() - start_time
print(f"\nAdaLoRA training time: {adalora_time:.1f}s")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 665,522 || all params: 110,149,336 || trainable%: 0.6042


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Matthews Correlation
1,1.178050,2.383596,0.188569
2,0.997825,2.536995,0.386026
3,0.954590,2.564013,0.406693


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



AdaLoRA training time: 314.5s


In [31]:
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import matthews_corrcoef
import numpy as np
import time

class SoRALayer(nn.Module):
    def __init__(self, in_features, out_features, r=8, lora_alpha=16, lambda_reg=0.01):
        super().__init__()
        self.r = r
        self.lora_alpha = lora_alpha
        self.lambda_reg = lambda_reg
        
        # Low rank matrices
        self.A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, r))
        
        # Gating vector — this is what makes it SoRA
        self.g = nn.Parameter(torch.ones(r))
        
        self.scaling = lora_alpha / r
    
    def forward(self, x):
        # Apply gating to rank components
        g_clamped = self.g.clamp(min=0)
        return (x @ (self.B * g_clamped.unsqueeze(0)) @ self.A).T * self.scaling
    
    def proximal_update(self, lr):
        # Soft thresholding — drives small gates to zero
        with torch.no_grad():
            self.g.data = torch.sign(self.g.data) * torch.clamp(
                self.g.data.abs() - self.lambda_reg * lr, min=0
            )
    
    def effective_rank(self):
        return (self.g.abs() > 1e-6).sum().item()

class SoRABERT(nn.Module):
    def __init__(self, base_model, r=8, lora_alpha=16, lambda_reg=0.01):
        super().__init__()
        self.base = base_model
        self.sora_layers = nn.ModuleDict()
        
        # Freeze base model
        for param in self.base.parameters():
            param.requires_grad = False
        
        # Add SoRA to attention layers
        for i in range(12):
            hidden = 768
            prefix = f"layer_{i}"
            self.sora_layers[f"{prefix}_q"] = SoRALayer(hidden, hidden, r, lora_alpha, lambda_reg)
            self.sora_layers[f"{prefix}_v"] = SoRALayer(hidden, hidden, r, lora_alpha, lambda_reg)
        
        # Classifier head
        self.classifier = nn.Linear(hidden, 2)
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.base.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(pooled)
        
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        
        from transformers.modeling_outputs import SequenceClassifierOutput
        return SequenceClassifierOutput(loss=loss, logits=logits)
    
    def get_trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Load fresh BERT
base_model3 = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
sora_model = SoRABERT(base_model3).cuda()
print(f"SoRA trainable params: {sora_model.get_trainable_params():,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SoRA trainable params: 296,642


In [32]:
class SoRATrainer(Trainer):
    def training_step(self, model, inputs, num_items_in_batch=None):
        loss = super().training_step(model, inputs, num_items_in_batch)
        lr = self.args.learning_rate
        actual_model = model.module if hasattr(model, 'module') else model
        for layer in actual_model.sora_layers.values():
            layer.proximal_update(lr)
        return loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    mcc = matthews_corrcoef(labels, predictions)
    return {"matthews_correlation": mcc}

training_args3 = TrainingArguments(
    output_dir="./sora_cola",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.01,
)

start_time = time.time()
trainer3 = SoRATrainer(
    model=sora_model,
    args=training_args3,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer3.train()
sora_time = time.time() - start_time
print(f"\nSoRA training time: {sora_time:.1f}s")

# Check effective ranks
for name, layer in sora_model.sora_layers.items():
    print(f"{name} effective rank: {layer.effective_rank()}/8")
    

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
import numpy as np

def proximal_update_numpy(g, lr, lambda_reg):
    # Soft thresholding
    # Step 1: subtract gradient step (g stays same for L1)
    # Step 2: apply soft threshold
    return np.sign(g) * np.maximum(np.abs(g) - lambda_reg * lr, 0)

def sgd_l1_numpy(g, lr, lambda_reg):
    # SGD with L1 subgradient
    # subgradient of |g| is sign(g)
    return g - lr * lambda_reg * np.sign(g)

# Test both
g = np.array([0.5, -0.3, 0.01, -0.001, 0.8])
lr = 0.1
lambda_reg = 0.05

g_prox = proximal_update_numpy(g, lr, lambda_reg)
g_sgd = sgd_l1_numpy(g, lr, lambda_reg)

print("Original g:  ", g)
print("Proximal g:  ", g_prox)
print("SGD L1 g:    ", g_sgd)
print("\nZeros in proximal:", (g_prox == 0).sum())
print("Zeros in SGD:     ", (g_sgd == 0).sum())

In [ ]:
import torch

def proximal_update_torch(g, lr, lambda_reg):
    return torch.sign(g) * torch.clamp(g.abs() - lambda_reg * lr, min=0)

def sgd_l1_torch(g, lr, lambda_reg):
    return g - lr * lambda_reg * torch.sign(g)

# Test
g = torch.tensor([0.5, -0.3, 0.01, -0.001, 0.8])
g_prox = proximal_update_torch(g, lr=0.1, lambda_reg=0.05)
g_sgd = sgd_l1_torch(g, lr=0.1, lambda_reg=0.05)

print("Original g:  ", g)
print("Proximal g:  ", g_prox)
print("SGD L1 g:    ", g_sgd)
print("\nZeros in proximal:", (g_prox == 0).sum().item())
print("Zeros in SGD:     ", (g_sgd == 0).sum().item())

In [ ]:
import torch
import torch.nn as nn

class SimplifiedMambaLoRA(nn.Module):
    """
    Simplified Mamba-style SSM with LoRA adaptation on B and C matrices.
    Demonstrates how LoRA would be applied in a state space model.
    """
    def __init__(self, d_model=64, d_state=16, r=4, lora_alpha=8):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        
        # Core SSM matrices (frozen in practice)
        self.A = nn.Parameter(torch.randn(d_state))
        self.B = nn.Parameter(torch.randn(d_model, d_state))
        self.C = nn.Parameter(torch.randn(d_state, d_model))
        self.D = nn.Parameter(torch.ones(d_model))
        
        # Freeze base matrices
        for p in [self.A, self.B, self.C, self.D]:
            p.requires_grad = False
        
        # LoRA on B matrix: B + B_lora where B_lora = B_b @ B_a
        self.B_a = nn.Parameter(torch.randn(r, d_state) * 0.01)
        self.B_b = nn.Parameter(torch.zeros(d_model, r))
        
        # LoRA on C matrix: C + C_lora where C_lora = C_b @ C_a  
        self.C_a = nn.Parameter(torch.randn(r, d_model) * 0.01)
        self.C_b = nn.Parameter(torch.zeros(d_state, r))
        
        self.scaling = lora_alpha / r
    
    def forward(self, x):
        # B and C with LoRA
        B_adapted = self.B + (self.B_b @ self.B_a) * self.scaling
        C_adapted = self.C + (self.C_b @ self.C_a) * self.scaling
        
        # Simple SSM step
        h = torch.tanh(x @ B_adapted)
        y = h @ C_adapted + x * self.D
        return y
    
    def trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Test it
model = SimplifiedMambaLoRA()
x = torch.randn(2, 10, 64)
out = model(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Trainable params: {model.trainable_params()}")
print(f"Total params: {sum(p.numel() for p in model.parameters())}")

In [ ]:
class SimplifiedxLSTMLoRA(nn.Module):
    """
    Simplified xLSTM with LoRA on gate matrices.
    Demonstrates how LoRA would be applied to recurrent architectures.
    """
    def __init__(self, input_size=64, hidden_size=64, r=4, lora_alpha=8):
        super().__init__()
        self.hidden_size = hidden_size
        self.scaling = lora_alpha / r
        
        # Core gate matrices (frozen)
        self.W_i = nn.Parameter(torch.randn(input_size, hidden_size))  # input gate
        self.W_f = nn.Parameter(torch.randn(input_size, hidden_size))  # forget gate
        self.W_o = nn.Parameter(torch.randn(input_size, hidden_size))  # output gate
        self.W_c = nn.Parameter(torch.randn(input_size, hidden_size))  # cell update
        
        # Freeze base matrices
        for p in [self.W_i, self.W_f, self.W_o, self.W_c]:
            p.requires_grad = False
        
        # LoRA on input and forget gates (most important for adaptation)
        self.W_i_a = nn.Parameter(torch.randn(r, hidden_size) * 0.01)
        self.W_i_b = nn.Parameter(torch.zeros(input_size, r))
        
        self.W_f_a = nn.Parameter(torch.randn(r, hidden_size) * 0.01)
        self.W_f_b = nn.Parameter(torch.zeros(input_size, r))
        
        self.bias = nn.Parameter(torch.zeros(hidden_size))
    
    def forward(self, x):
        B, T, _ = x.shape
        h = torch.zeros(B, self.hidden_size, device=x.device)
        c = torch.zeros(B, self.hidden_size, device=x.device)
        outputs = []
        
        W_i_adapted = self.W_i + (self.W_i_b @ self.W_i_a) * self.scaling
        W_f_adapted = self.W_f + (self.W_f_b @ self.W_f_a) * self.scaling
        
        for t in range(T):
            xt = x[:, t, :]
            i = torch.sigmoid(xt @ W_i_adapted)
            f = torch.sigmoid(xt @ W_f_adapted)
            o = torch.sigmoid(xt @ self.W_o)
            g = torch.tanh(xt @ self.W_c)
            c = f * c + i * g
            h = o * torch.tanh(c)
            outputs.append(h.unsqueeze(1))
        
        return torch.cat(outputs, dim=1)
    
    def trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Test it
xlstm_model = SimplifiedxLSTMLoRA()
x = torch.randn(2, 10, 64)
out = xlstm_model(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Trainable params: {xlstm_model.trainable_params()}")
print(f"Total params: {sum(p.numel() for p in xlstm_model.parameters())}")